In [24]:
# ============================================================================
# COMPLETE ANOVA ANALYSIS CODE FOR FLY MASS AND LENGTH DATA
# ============================================================================
# This code performs:
# 1. Male vs Female comparison within each group
# 2. Male vs Female total
# 3. All pairwise t-tests between groups
# ============================================================================

import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

# ============================================================================
# STEP 1: LOAD YOUR DATA
# ============================================================================

raw_data = {
    'Axenic_early': {'Male': [0.743, 0.774, 0.732, 0.715], 'Female': [1.089, 1.149, 1.147, 1.1]},
    'Axenic_adult': {'Male': [0.788, 0.704, 0.854], 'Female': [1.206, 1.378, 1.3]},
    'Bacteria_early': {'Male': [0.953, 0.941, 0.861], 'Female': [1.383, 1.332, 1.478, 0.985]},
    'Reintroduce': {'Male': [0.748, 0.582, 0.751], 'Female': [1.253, 0.84, 1.171]},
    'Bacteria_adult': {'Male': [0.881, 0.73, 0.852], 'Female': [1.531, 1.124, 1.181]}
}

length_data = {
    'Axenic_early': {'Male': [2.568, 2.516, 2.574, 2.425, 2.09, 2.425, 2.626, 2.495, 2.564, 2.567, 2.434, 2.361, 2.323, 2.366, 2.426, 2.506, 2.23, 2.229, 2.376, 2.403], 'Female': [2.52, 2.672, 2.548, 2.658, 2.71, 2.689, 2.793, 2.734, 2.627, 2.691, 2.605, 3.035, 2.652, 2.967, 2.949, 2.8, 2.619, 3.164, 3.059, 2.969]},
    'Axenic_adult': {'Male': [2.108, 2.214, 2.019, 2.175, 2.188, 2.247, 2.43, 2.35, 2.338, 2.4, 2.586, 2.483, 2.488, 2.385, 2.395], 'Female': [2.723, 2.586, 2.641, 2.801, 2.9, 2.889, 2.972, 2.994, 3.059, 2.991, 2.758, 2.88, 2.86, 2.785, 2.798]},
    'Bacteria_early': {'Male': [2.582, 2.406, 2.519, 2.622, 2.559, 2.521, 2.63, 2.581, 2.524, 2.563, 2.235, 2.378, 2.379, 2.437, 2.424], 'Female': [2.875, 2.945, 2.902, 2.692, 2.822, 3.05, 3.008, 3.151, 3.167, 2.94, 2.768, 2.925, 2.869, 2.923, 2.865, 2.59, 2.818, 2.89, 2.59, 2.769]},
    'Reintroduce': {'Male': [2.056, 2.244, 2.312, 2.083, 2.197, 2.076, 2.07, 1.951, 1.962, 1.964, 2.282, 2.041, 2.09, 2.354, 1.958], 'Female': [2.57, 2.907, 2.458, 2.368, 2.701, 2.585, 2.468, 2.464, 2.476, 2.499, 2.393, 2.457, 2.798, 2.524, 2.848]},
    'Bacteria_adult': {'Male': [2.193, 2.24, 2.134, 2.537, 2.293, 2.575, 2.141, 2.237, 2.132, 2.338, 2.237, 2.132, 2.033, 2.246, 2.311], 'Female': [2.705, 2.896, 3.142, 2.602, 2.528, 2.819, 2.656, 2.798, 2.66, 2.935, 2.932, 2.733, 2.844, 2.983, 2.818]}
}

# Create dataframe
rows = []
for group in raw_data.keys():
    for sex in ['Male', 'Female']:
        weights = raw_data[group][sex]
        lengths = length_data[group][sex]
        for i, (w, l) in enumerate(zip(weights, lengths)):
            rows.append({'Group': group, 'Sex': sex, 'Weight': w, 'Length': l})

df = pd.DataFrame(rows)
groups = ['Axenic_early', 'Axenic_adult', 'Bacteria_early', 'Reintroduce', 'Bacteria_adult']

# ============================================================================
# STEP 2: MALE vs FEMALE COMPARISON (within each group)
# ============================================================================
# =========
# For weight,
#Mann‑Whitney U doesn't really work for weight. 
# When both groups have only 3 observations, 
# the smallest possible two‑sided p‑value is 0.1. 
# Total number of ways to assign 6 values into two groups of 3 = choose(6,3) = 20. 
# If all values of group A are smaller than all values of group B, the U statistic is 0. 
# The probability of observing U = 0 under the null is 1/20 = 0.05. For a two‑sided test, 
# you double that → p = 0.1. p < 0.05 mean significant, it will not work.
# =========

groups = list(raw_data.keys())
rows = []
for group in groups:
    for sex in ["Male", "Female"]:
        for l in raw_data[group][sex]:
            rows.append({"Group": group, "Sex": sex, "weight": l})

df_weight = pd.DataFrame(rows)

print("\n--- WEIGHT ONLY ---")
print(f"{'Group':<20} {'Male':>8} {'Female':>8} {'p-value':>10} {'Sig':>5}")
print('-' * 55)
for group in groups:
    male = df_weight[(df_weight['Group']==group) & (df_weight['Sex']=='Male')]['weight']
    female = df_weight[(df_weight['Group']==group) & (df_weight['Sex']=='Female')]['weight']
    # u_stat_weight, p = stats.mannwhitneyu(male, female, alternative='two-sided')
    t, p = stats.ttest_ind(male, female)
    sig = '*' if p < 0.05 else ''
    print(f"{group:<20} {male.mean():>8.3f} {female.mean():>8.3f} {p:>10.4f} {sig:>5}")

print("OVERALL MALE vs FEMALE (pooled across all groups)")
male_weight = df_weight[df_weight['Sex'] == 'Male']['weight']
female_weight = df_weight[df_weight['Sex'] == 'Female']['weight']
t_weight, p_weight = stats.ttest_ind(male_weight, female_weight, equal_var=False)  # Welch's test
#u_stat_weight, p_weight = stats.mannwhitneyu(male_weight, female_weight, alternative='two-sided')
print(f"\nWEIGHT (pooled): Male n={len(male_weight)}, mean={male_weight.mean():.3f}; Female n={len(female_weight)}, mean={female_weight.mean():.3f}")
print(f"  t = {t_weight:.3f}, p = {p_weight:.4f} {'*' if p_weight < 0.05 else ''}")

# =========
# For Length
# =========
groups = list(length_data.keys())
rows = []
for group in groups:
    for sex in ["Male", "Female"]:
        for l in length_data[group][sex]:
            rows.append({"Group": group, "Sex": sex, "Length": l})

df_length = pd.DataFrame(rows)

print("\n--- LENGTH ONLY ---")
print(f"{'Group':<20} {'Male':>8} {'Female':>8} {'p-value':>10} {'Sig':>5}")
print('-' * 55)
for group in groups:
    male = df_length[(df_length['Group']==group) & (df_length['Sex']=='Male')]['Length']
    female = df_length[(df_length['Group']==group) & (df_length['Sex']=='Female')]['Length']
    t, p = stats.ttest_ind(male, female, equal_var=False)  # Welch's t-test handles unequal n
    sig = '*' if p < 0.05 else ''
    print(f"{group:<20} {male.mean():>8.3f} {female.mean():>8.3f} {p:>10.4f} {sig:>5}")

print("OVERALL MALE vs FEMALE (pooled across all groups)")
male_length = df_length[df_length['Sex'] == 'Male']['Length']
female_length = df_length[df_length['Sex'] == 'Female']['Length']
u_stat_length, p_length = stats.mannwhitneyu(male_length, female_length, alternative='two-sided')
t_length, p = stats.ttest_ind(male_length, female_length, equal_var=False)
print(f"\nLENGTH (pooled): Male n={len(male_length)}, mean={male_length.mean():.3f}; Female n={len(female_length)}, mean={female_length.mean():.3f}")
print(f" t={t_length:.3f}, u = {u_stat_length:.3f}, pt = {p:.4f}, p = {p_length:.4f} {'*' if p_length < 0.05 else ''}")

# ============================================================================
# STEP 3: PAIRWISE T-TESTS (all group comparisons)
# ============================================================================
print("\n" + "=" * 70)
print("3. PAIRWISE T-TESTS")
print("=" * 70)

for sex in ['Male', 'Female']:
    for measure in ['Weight', 'Length']:
        print(f"\n--- {sex.upper()}S - {measure.upper()} ---")
        sex_data = df[df['Sex'] == sex]
        
        for g1, g2 in combinations(groups, 2):
            data1 = sex_data[sex_data['Group'] == g1][measure]
            data2 = sex_data[sex_data['Group'] == g2][measure]
            t, p = stats.ttest_ind(data1, data2)
            sig = '*' if p < 0.05 else ''
            if p < 0.05:  # Only print significant results
                print(f"  {g1} vs {g2}: p = {p:.4f} {sig}")

print("\n" + "=" * 70)
print("DONE! (* indicates p < 0.05)")
print("=" * 70)

print("3. PAIRWISE U-TESTS (Mann-Whitney U)")

for sex in ['Male', 'Female']:
    for measure in ['Weight', 'Length']:
        print(f"\n--- {sex.upper()}S - {measure.upper()} ---")
        sex_data = df[df['Sex'] == sex]
        
        for g1, g2 in combinations(groups, 2):
            data1 = sex_data[sex_data['Group'] == g1][measure]
            data2 = sex_data[sex_data['Group'] == g2][measure]
            u_stat, p = stats.mannwhitneyu(data1, data2, alternative='two-sided')
            print(f"  {g1} vs {g2}: p = {p:.4f} {sig}")

# ============================================================================
# STEP 5: MW tes leat (all group comparisons) SEARCH to DO Later
# ============================================================================
# print("\n" + "=" * 70)
# print("3. PAIRWISE T-TESTS")
# print("=" * 70)

# for sex in ['Male', 'Female']:
#     for measure in ['Weight', 'Length']:
#         print(f"\n--- {sex.upper()}S - {measure.upper()} ---")
#         sex_data = df[df['Sex'] == sex]
        
#         for g1, g2 in combinations(groups, 2):
#             data1 = sex_data[sex_data['Group'] == g1][measure]
#             data2 = sex_data[sex_data['Group'] == g2][measure]
#             t, p = stats.ttest_ind(data1, data2)
#             sig = '*' if p < 0.05 else ''
#             if p < 0.05:  # Only print significant results
#                 print(f"  {g1} vs {g2}: p = {p:.4f} {sig}")

# print("\n" + "=" * 70)
# print("DONE! (* indicates p < 0.05)")
# print("=" * 70)


--- WEIGHT ONLY ---
Group                    Male   Female    p-value   Sig
-------------------------------------------------------
Axenic_early            0.741    1.121     0.0000     *
Axenic_adult            0.782    1.295     0.0015     *
Bacteria_early          0.918    1.294     0.0336     *
Reintroduce             0.694    1.088     0.0461     *
Bacteria_adult          0.821    1.279     0.0278     *
OVERALL MALE vs FEMALE (pooled across all groups)

WEIGHT (pooled): Male n=16, mean=0.788; Female n=17, mean=1.215
  t = -8.741, p = 0.0000 *

--- LENGTH ONLY ---
Group                    Male   Female    p-value   Sig
-------------------------------------------------------
Axenic_early            2.425    2.773     0.0000     *
Axenic_adult            2.320    2.842     0.0000     *
Bacteria_early          2.491    2.878     0.0000     *
Reintroduce             2.109    2.568     0.0000     *
Bacteria_adult          2.252    2.803     0.0000     *
OVERALL MALE vs FEMALE (pooled a

In [1]:
#!/usr/bin/env python3
"""
Body Size Analysis Code
=======================
Load your mass_length.xlsx and analyze Male vs Female within each treatment group.

Author: For Tian (Ludington Lab)
"""

import pandas as pd
import numpy as np
from scipy import stats

# =============================================================================
# 1. LOAD YOUR DATA
# =============================================================================
file_path = 'mass_length.xlsx'  # Change this to your file path

# Load sheets
length_df = pd.read_excel(file_path, sheet_name='Length')
mass_df = pd.read_excel(file_path, sheet_name='weight_GF_B_GFB')

print("Columns in Length sheet:", length_df.columns.tolist())
print("Columns in Mass sheet:", mass_df.columns.tolist()[:11])

# =============================================================================
# 2. DEFINE COLUMNS
# =============================================================================
length_cols = [
    'axenic early ecolsion M', 'axenic early ecolsion F',
    'axenic adult M', 'axenic adult F',
    'Bacteria early ecolsion M', 'Bacteria early ecolsion F',
    'Reintroduce M', 'Reintroduce F',
    'Bacteria adult M', 'Bacteria adult F'
]

# =============================================================================
# 3. EXTRACT DATA
# =============================================================================
# Length data - remove outliers < 1.5 mm
length_data = {}
for col in length_cols:
    values = pd.to_numeric(length_df[col], errors='coerce').dropna()
    values = values[values >= 1.5]  # Remove outliers
    length_data[col] = values
    print(f"Length - {col}: n = {len(values)}")

# Mass data - only first 4 rows have actual data
mass_data = {}
for col in length_cols:
    if col in mass_df.columns:
        values = pd.to_numeric(mass_df[col].iloc[:4], errors='coerce').dropna()
        mass_data[col] = values
        print(f"Mass - {col}: n = {len(values)}")

# =============================================================================
# 4. COMPARE MALE vs FEMALE WITHIN EACH GROUP
# =============================================================================
def compare_male_female(male_data, female_data, group_name, measure=''):
    """Compare male vs female using Mann-Whitney U test"""
    stat, pval = stats.mannwhitneyu(male_data, female_data, alternative='two-sided')
    diff = female_data.mean() - male_data.mean()
    pct = (female_data.mean() / male_data.mean() - 1) * 100

    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'

    print(f"\n{group_name} ({measure}):")
    print(f"  Males:   n={len(male_data):2d}, mean={male_data.mean():.3f}")
    print(f"  Females: n={len(female_data):2d}, mean={female_data.mean():.3f}")
    print(f"  Diff (F-M): {diff:.3f} ({pct:.1f}%)")
    print(f"  Mann-Whitney U = {stat:.1f}, p = {pval:.4f} {sig}")

# Groups to compare
groups = [
    ('axenic early ecolsion', 'Axenic Early Eclosion'),
    ('axenic adult', 'Axenic Adult (B2_conA)'),
    ('Bacteria early ecolsion', 'Bacteria Early Eclosion (A_inoA)'),
    ('Reintroduce', 'Reintroduce (B1_reintro)'),
    ('Bacteria adult', 'Bacteria Adult'),
]

print("\n" + "="*60)
print("BODY LENGTH COMPARISONS")
print("="*60)
for col_prefix, name in groups:
    compare_male_female(length_data[f'{col_prefix} M'], 
                       length_data[f'{col_prefix} F'], 
                       name, 'Length mm')

print("\n" + "="*60)
print("BODY MASS COMPARISONS")
print("="*60)
for col_prefix, name in groups:
    if f'{col_prefix} M' in mass_data and f'{col_prefix} F' in mass_data:
        if len(mass_data[f'{col_prefix} M']) >= 2 and len(mass_data[f'{col_prefix} F']) >= 2:
            compare_male_female(mass_data[f'{col_prefix} M'], 
                               mass_data[f'{col_prefix} F'], 
                               name, 'Mass mg')

# =============================================================================
# 5. OPTIONAL: POOL GROUPS AND COMPARE
# =============================================================================
print("\n" + "="*60)
print("POOLED ANALYSIS: AXENIC vs BACTERIA")
print("="*60)

# Pool axenic (early + adult)
axenic_M = pd.concat([length_data['axenic early ecolsion M'], 
                      length_data['axenic adult M']])
axenic_F = pd.concat([length_data['axenic early ecolsion F'], 
                      length_data['axenic adult F']])

# Pool bacteria (early + adult) - EXCLUDING reintroduce
bacteria_M = pd.concat([length_data['Bacteria early ecolsion M'], 
                        length_data['Bacteria adult M']])
bacteria_F = pd.concat([length_data['Bacteria early ecolsion F'], 
                        length_data['Bacteria adult F']])

print("\nPooled Axenic (early + adult):")
compare_male_female(axenic_M, axenic_F, 'Axenic Pooled', 'Length mm')

print("\nPooled Bacteria (early + adult, excluding reintroduce):")
compare_male_female(bacteria_M, bacteria_F, 'Bacteria Pooled', 'Length mm')

# Compare Axenic vs Bacteria within each sex
print("\n" + "="*60)
print("AXENIC vs BACTERIA (within sex)")
print("="*60)

stat, pval = stats.mannwhitneyu(axenic_M, bacteria_M, alternative='two-sided')
print(f"\nMALES: Axenic (n={len(axenic_M)}, mean={axenic_M.mean():.3f}) vs Bacteria (n={len(bacteria_M)}, mean={bacteria_M.mean():.3f})")
print(f"  Mann-Whitney U = {stat:.1f}, p = {pval:.4f}")

stat, pval = stats.mannwhitneyu(axenic_F, bacteria_F, alternative='two-sided')
print(f"\nFEMALES: Axenic (n={len(axenic_F)}, mean={axenic_F.mean():.3f}) vs Bacteria (n={len(bacteria_F)}, mean={bacteria_F.mean():.3f})")
print(f"  Mann-Whitney U = {stat:.1f}, p = {pval:.4f}")


Columns in Length sheet: ['Unnamed: 0', 'axenic early ecolsion M', 'axenic early ecolsion F', 'axenic adult M', 'axenic adult F', 'Bacteria early ecolsion M', 'Bacteria early ecolsion F', 'Reintroduce M', 'Reintroduce F', 'Bacteria adult M', 'Bacteria adult F']
Columns in Mass sheet: ['Unnamed: 0', 'axenic early ecolsion M', 'axenic early ecolsion F', 'axenic adult M', 'axenic adult F', 'Bacteria early ecolsion M', 'Bacteria early ecolsion F', 'Reintroduce M', 'Reintroduce F', 'Bacteria adult M', 'Bacteria adult F']
Length - axenic early ecolsion M: n = 23
Length - axenic early ecolsion F: n = 23
Length - axenic adult M: n = 18
Length - axenic adult F: n = 18
Length - Bacteria early ecolsion M: n = 18
Length - Bacteria early ecolsion F: n = 21
Length - Reintroduce M: n = 16
Length - Reintroduce F: n = 16
Length - Bacteria adult M: n = 16
Length - Bacteria adult F: n = 16
Mass - axenic early ecolsion M: n = 4
Mass - axenic early ecolsion F: n = 4
Mass - axenic adult M: n = 3
Mass - axen